<a href="https://colab.research.google.com/github/Sanjay-7793/Village_growth/blob/main/VillageGrowth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary spatial libraries
!pip install geopandas geemap rasterio shapely requests -q



In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
import json
import time
from shapely import wkt

In [ ]:
# Authenticate and Initialize Google Earth Engine
# This will prompt you to click a link and generate an auth token.
ee.Authenticate()
ee.Initialize(project='GEE account')

In [ ]:
# ==========================================
# 1. LOAD & FILTER NWIC GEOJSON
# ==========================================
print("Loading NWIC Village Boundaries...")

# Path to the file you uploaded to Colab
file_path = '/content/vb_soi_ka.GeoJSON'
gdf = gpd.read_file(file_path)


col_district = 'district'     # district name in attribute table
col_village = 'village'     # Village name in attribute table
col_block = 'block'       # Block name in attribute table
col_village_code = 'vlcode' # unique code in attribute table

# Clean the district column to ensure capitalization matches
gdf[col_district] = gdf[col_district].str.title()

target_districts = ['Mysuru', 'Mandya', 'Hassan', 'Raichur', 'Yadgiri', 'Tumakuru']

filtered_gdf = gdf[gdf[col_district].isin(target_districts)].copy()
print(f"Filtered down to {len(filtered_gdf)} villages in target districts.")

# Calculate Polygon Area (Requires metric CRS like EPSG:32643 for accurate meters)
filtered_gdf = filtered_gdf.to_crs(epsg=32643)
filtered_gdf['village_area_sqm'] = filtered_gdf.geometry.area

# Reproject back to WGS84 (Lat/Lon) for Earth Engine and OSM APIs
filtered_gdf = filtered_gdf.to_crs(epsg=4326)

# Create a clean unique ID for the merge later
filtered_gdf['uid'] = range(1, len(filtered_gdf) + 1)

# Convert to Earth Engine FeatureCollection
village_fc = geemap.geopandas_to_ee(filtered_gdf)

# ==========================================
# 2. EARTH ENGINE RASTER PROCESSING
# ==========================================
print("Processing Earth Engine spatial aggregations (LULC & Nightlights)...")

def get_class_area(image, class_value, band_name):
    # Create binary mask and multiply by pixel area (square meters)
    return image.eq(class_value).multiply(ee.Image.pixelArea()).rename(band_name)

# --- 2021 DATA (Dynamic World Fallback for 2021 LULC) ---
dw_2021 = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
            .filterDate('2021-01-01', '2021-12-31').mode().select('label')
crop_2021 = get_class_area(dw_2021, 4, 'crop_area_21')
built_2021 = get_class_area(dw_2021, 6, 'built_area_21')
waste_2021 = get_class_area(dw_2021, 7, 'waste_area_21')

viirs_2021 = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
               .filterDate('2021-01-01', '2021-12-31').median().select(['avg_rad'], ['light_21'])

# --- 2025 DATA (Dynamic World Fallback for 2025 LULC) ---
# Classes: 4=Crop, 6=Built, 7=Bare
dw_2025 = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1") \
            .filterDate('2025-01-01', '2025-12-31').mode().select('label')
crop_2025 = get_class_area(dw_2025, 4, 'crop_area_25')
built_2025 = get_class_area(dw_2025, 6, 'built_area_25')
waste_2025 = get_class_area(dw_2025, 7, 'waste_area_25')

viirs_2025 = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG") \
               .filterDate('2025-01-01', '2025-12-31').median().select(['avg_rad'], ['light_25'])

# Combine all bands
raster_stack = ee.Image([
    crop_2021, built_2021, waste_2021, viirs_2021,
    crop_2025, built_2025, waste_2025, viirs_2025
])

# --- THE CHUNKING LOGIC ---for avoiding timeout error
import math

chunk_size = 1000 # Process 1000 villages at a time to stay under the 10MB limit
num_chunks = math.ceil(len(filtered_gdf) / chunk_size)
all_ee_dfs = []

print(f"Splitting data into {num_chunks} chunks to avoid GEE payload limits...")

for i in range(num_chunks):
    print(f"Processing chunk {i+1} of {num_chunks}...")

    # Slice the GeoDataFrame
    chunk_gdf = filtered_gdf.iloc[i*chunk_size : (i+1)*chunk_size]

    # Convert only this chunk to an Earth Engine FeatureCollection
    chunk_fc = geemap.geopandas_to_ee(chunk_gdf)

    # Run Zonal Statistics for this chunk
    stats_fc = raster_stack.reduceRegions(
        collection=chunk_fc,
        reducer=ee.Reducer.sum().combine(ee.Reducer.median(), sharedInputs=True),
        scale=10
    )

    # Convert GEE results to Pandas
    chunk_df = geemap.ee_to_df(stats_fc)
    all_ee_dfs.append(chunk_df)

# Concatenate all processed chunks back into a single DataFrame
ee_df = pd.concat(all_ee_dfs, ignore_index=True)
print("Earth Engine processing complete!")




Loading NWIC Village Boundaries...
Filtered down to 9068 villages in target districts.
Processing Earth Engine spatial aggregations (LULC & Nightlights)...
Splitting data into 10 chunks to avoid GEE payload limits...
Processing chunk 1 of 10...
Processing chunk 2 of 10...
Processing chunk 3 of 10...
Processing chunk 4 of 10...
Processing chunk 5 of 10...
Processing chunk 6 of 10...
Processing chunk 7 of 10...
Processing chunk 8 of 10...
Processing chunk 9 of 10...
Processing chunk 10 of 10...
Earth Engine processing complete!


In [ ]:
# ==========================================
# 3. HISTORICAL OSM DATA extractor
# ==========================================
print("Cleaning geometries and fetching historical OSM road networks...")
import math
import time

url = "https://api.ohsome.org/v1/elements/length/groupBy/boundary"

# make_valid() fixes self-intersecting polygons,
# simplify(0.0005) removes microscopic vertices (roughly 50-meter tolerance).
safe_gdf = filtered_gdf.copy()
safe_gdf['geometry'] = safe_gdf['geometry'].make_valid().simplify(0.0005)

def fetch_batch_roads(gdf_chunk, date_str, retries=2):
    geojson_payload = gdf_chunk.set_index('uid')[['geometry']].to_json()

    payload = {
        "bpolys": geojson_payload,
        "filter": "highway=*",
        "time": date_str
    }

    for attempt in range(retries):
        try:
            response = requests.post(url, data=payload, timeout=60)
            if response.status_code == 200:
                res_json = response.json()
                road_dict = {}
                for item in res_json.get('groupByResult', []):
                    uid = int(item['groupByObject'])
                    length = item['result'][0]['value']
                    road_dict[uid] = length
                return road_dict
            else:
                print(f"  [Attempt {attempt+1}] API Error {response.status_code} for {date_str}.")
                time.sleep(2) # Wait before retrying
        except Exception as e:
             print(f"  [Attempt {attempt+1}] Request failed: {e}")
             time.sleep(2)

    print(f"  -> Failed to fetch {date_str} for this chunk after {retries} attempts. Filling with 0.")
    return {} # Return empty dict if all retries fail so the pipeline doesn't crash

# Dropping from 500 to 100 prevents Ohsome's backend from timing out on heavy districts.
chunk_size = 500
num_chunks = math.ceil(len(safe_gdf) / chunk_size)

dict_2021 = {}
dict_2025 = {}

for i in range(num_chunks):
    print(f"Querying OSM chunk {i+1} of {num_chunks}...")
    chunk = safe_gdf.iloc[i*chunk_size : (i+1)*chunk_size]

    dict_2021.update(fetch_batch_roads(chunk, '2021-01-01'))
    dict_2025.update(fetch_batch_roads(chunk, '2025-01-01'))

    time.sleep(1.5) # reduce load on API server


filtered_gdf['road_length_21'] = filtered_gdf['uid'].map(dict_2021).fillna(0)
filtered_gdf['road_length_25'] = filtered_gdf['uid'].map(dict_2025).fillna(0)

print("Batch OSM extraction complete!")

Cleaning geometries and fetching historical OSM road networks...
Querying OSM chunk 1 of 19...
Querying OSM chunk 2 of 19...
Querying OSM chunk 3 of 19...
Querying OSM chunk 4 of 19...
Querying OSM chunk 5 of 19...
Querying OSM chunk 6 of 19...
Querying OSM chunk 7 of 19...
Querying OSM chunk 8 of 19...
Querying OSM chunk 9 of 19...
Querying OSM chunk 10 of 19...
Querying OSM chunk 11 of 19...
Querying OSM chunk 12 of 19...
Querying OSM chunk 13 of 19...
Querying OSM chunk 14 of 19...
Querying OSM chunk 15 of 19...
Querying OSM chunk 16 of 19...
Querying OSM chunk 17 of 19...
Querying OSM chunk 18 of 19...
Querying OSM chunk 19 of 19...
Batch OSM extraction complete!


In [ ]:
# ==========================================
# 4. FINAL MERGE AND FORMATTING
# ==========================================
print("Merging datasets and exporting to CSV...")

gee_columns = [
    'uid',
    'crop_area_21_sum', 'waste_area_21_sum', 'built_area_21_sum', 'light_21_median',
    'crop_area_25_sum', 'waste_area_25_sum', 'built_area_25_sum', 'light_25_median'
]

# Ensure we only extract columns that actually successfully generated in GEE
valid_gee_columns = [col for col in gee_columns if col in ee_df.columns]
ee_df_clean = ee_df[valid_gee_columns]

# Now merge. Because there are no duplicate column names, Pandas will NOT create '_x' suffixes!
final_df = pd.merge(filtered_gdf, ee_df_clean, on='uid', how='left')

# Define our output mapping
output_columns = {
    col_village_code: 'Village_Code',
    col_village: 'Village_Name',
    col_block: 'Block_Taluka_Name',
    col_district: 'District_Name',
    'village_area_sqm': 'Polygon_Area_sqm',
    'crop_area_21_sum': 'Crop_Area_sqm_2021',
    'waste_area_21_sum': 'Waste_Area_sqm_2021',
    'built_area_21_sum': 'Built_Area_sqm_2021',
    'light_21_median': 'Median_Nightlight_2021',
    'road_length_21': 'OSM_Road_Length_m_2021',
    'crop_area_25_sum': 'Crop_Area_sqm_2025',
    'waste_area_25_sum': 'Waste_Area_sqm_2025',
    'built_area_25_sum': 'Built_Area_sqm_2025',
    'light_25_median': 'Median_Nightlight_2025',
    'road_length_25': 'OSM_Road_Length_m_2025'
}

# Safely rename and extract
# This checks if the column exists before renaming, preventing KeyErrors if a class was missing
final_cols = {k: v for k, v in output_columns.items() if k in final_df.columns}
final_csv = final_df[list(final_cols.keys())].rename(columns=final_cols)

# Fill any NaN values (e.g., if a village had no roads or no built-up area) with 0
final_csv = final_csv.fillna(0)

# Export to CSV
output_path = '/content/karnataka_village_economic_growth.csv'
final_csv.to_csv(output_path, index=False)

print(f"Pipeline Complete! Output successfully saved to: {output_path}")

Merging datasets and exporting to CSV...
Pipeline Complete! Output successfully saved to: /content/karnataka_village_economic_growth.csv
